# gromos-rs Python API — Design Mockup (Priority 3)

**Status: MOCKUP. None of this runs yet.** It is a sketch of the *intended* API so we can react to real code instead of abstract questions.

Each 🔴 **DECISION** box marks something still open. Read the code, then tell me which option feels right.

## The one big idea (your Q6 insight)

GROMOS already splits the world into two files:
- `.top` → **Topology**: what atoms/bonds/charges exist (no positions)
- `.cnf` → **Configuration**: where atoms are in space (positions, velocities, box)

So we keep them as **two independent objects**, and a **`System` = Topology + Configuration** is just the two combined. You can hold either alone, or combine them. That's the whole model.

---
## Part 1 — Three nouns: `Topology`, `Configuration`, `System`

All three are **Rust objects** exposed via pyo3 (your Q1: System is Rust).

In [ ]:
import gromos

# --- Hold them independently (matches the two files) ---
topo = gromos.Topology.from_file("protein.top")     # connectivity only, no coords
conf = gromos.Configuration.from_file("protein.cnf") # coords only, no connectivity

print(topo)   # Topology(n_atoms=648, charge=-3)
print(conf)   # Configuration(n_atoms=648, box=(3.1, 3.1, 3.1))

# --- Or combine into a System (topology + coordinates together) ---
system = gromos.System(topo, conf)   # validates: topo.n_atoms == conf.n_atoms
print(system) # System(n_atoms=648, charge=-3, box=(3.1, 3.1, 3.1))

# Convenience: load both files straight into a System
system = gromos.System.from_files("protein.top", "protein.cnf")

**This resolves Q6 and Q7 at once:**
- `Topology` and `Configuration` are first-class — hold them alone if you want.
- `System` is the combination. It is what you usually pass around.
- No more `EnhancedConfiguration`. `Configuration` *is* the coordinate object; you can reach it as `system.configuration`.

```python
system.topology       # -> Topology
system.configuration  # -> Configuration
system.positions      # -> numpy (N, 3), f64   (shortcut for system.configuration.positions)
system.charge         # -> int (from topology)
system.n_atoms        # -> int (authoritative, from topology)
```

---
## Part 2 — The easy path: “I already have GROMOS files”

Most users start here. No building, no binaries — just load, inspect, run.

In [ ]:
system = gromos.System.from_files("water_216.top", "water_216.cnf")

# inspect
print(system.n_atoms)          # 648
print(system.charge)           # 0
print(system.positions.shape)  # (648, 3)
print(system.positions.dtype)  # float64   (uniform everywhere — no more f32/f64 mix)

# slice / select
ows = system.select("a:OW")    # a sub-System? or just indices?  see DECISION below

🔴 **DECISION D1 — what does `select()` return?**
- (a) a numpy index array `[0, 3, 6, ...]` (simple, composable with numpy)
- (b) a sub-`System` view (richer, but what does a topology “view” mean?)

*My lean: (a) for now.* Selections are indices; richer slicing later.

---
## Part 3 — The algebra: combining systems **in space** (your Q4 / Q5)

This is the hard part you flagged. `protein + water` has to decide **where things go**. Here are the two models side by side so you can pick.

In [ ]:
# ============================================================
# MODEL A — EAGER:  + merges coordinates immediately
# ============================================================
# Each operand already has real coordinates. '+' concatenates them.
# PROBLEM: two systems both centered at origin -> atoms overlap in space.
# So eager '+' only makes sense if you place them first:

protein = gromos.System.from_files("protein.top", "protein.cnf")
water   = gromos.System.from_files("water.top",   "water.cnf")

combined = protein + water.translate([2.0, 0.0, 0.0])   # YOU place it. explicit, but tedious.
# combined now has real, possibly-clashing coordinates.

In [ ]:
# ============================================================
# MODEL B — LAZY:  + builds an assembly *plan*, placement happens later
# ============================================================
# '+' and '* N' just record "these belong together, this many times".
# No coordinates are touched until you call .pack() / .solvate().
# The placement algorithm (ran_box / sim_box) runs ONCE, at the end.

assembly = protein * 1 + water * 216      # an Assembly (a plan), NOT yet placed
print(assembly)  # Assembly(protein x1, water x216)  — no coordinates yet

system = assembly.pack(density=900, seed=42)   # NOW ran_box places everything -> System
print(system)    # System(n_atoms=..., box=...)  — real coordinates

🔴 **DECISION D2 — eager or lazy `+`?**

| | Eager (A) | Lazy (B) |
|---|---|---|
| `protein + water` returns | a placed `System` (may clash) | an `Assembly` plan |
| who places atoms | you (`.translate`) | one `.pack()`/`.solvate()` at the end |
| matches GROMOS pipeline | no | **yes** (ran_box/sim_box place everything at once) |
| spatial clashes | your problem | handled by ran_box |

*My strong lean: **Lazy (B)***. It matches how `ran_box`/`sim_box` actually work (they place molecules with real algorithms — your Q2 point), and it cleanly separates “what's in the system” from “where it is.” The cost: one extra concept, `Assembly`.

🔴 **DECISION D3 (your Q5) — what does `protein * 10` mean?**
- Under Lazy (B): trivial — it just sets count=10 in the plan; `ran_box` makes 10 placements at `.pack()` time. **No problem.**
- Under Eager (A): you'd have to place 10 copies yourself. Painful.

This is the strongest argument for Lazy: `* N` only makes sense if placement is deferred.

---
## Part 4 — Solvation (your Q2 / Q3)

Confirmed: the solvent is a `System`, and solvation uses the real GROMOS placement binaries in the background.

In [ ]:
# water is itself a System (topology + coordinates of one SPC molecule)
water = gromos.ForceField("54a7").solvent("SPC")   # -> System

protein = gromos.System.from_files("protein.top", "protein.cnf")

# .solvate takes a System, returns a System.
# In the background this calls sim_box (fixed box) or ran_box (target density),
# because those carry the real GROMOS solvation algorithms (your Q2 reasoning).
solvated = protein.solvate(water, n=2000, density=900, seed=42)
print(solvated)         # System(n_atoms=..., ...)
print(solvated.n_atoms) # solute + 2000*3

# neutralize: adds counter-ions (background: ion + com_top)
solvated = solvated.neutralize("Na+")
print(solvated.charge)  # 0

🔴 **DECISION D4 — `n` vs `density` in `.solvate()`**
- give `n=` → exact molecule count (uses `sim_box` into a known box)
- give `density=` → fill to target density (uses `ran_box`)
- give both? → error, or n wins?

*My lean: exactly one of `n`/`density` required; giving both is an error.*

---
## Part 5 — Building from scratch: the 8 binaries & the VSOMM case (your Q6)

The traditional binaries stay. `ForceField.molecule(sequence)` drives `make_top` (+ coordinate pipeline) in the background and hands you back a `System`.

In [ ]:
ff = gromos.ForceField("54a7")

# One molecule from a building-block sequence.
# Background: make_top -> pdb2g96 -> gca -> gch  (the coordinate pipeline)
# Returns a System (topology AND coordinates together).
mol = ff.molecule(["NH3+", "ALA", "GLY", "COO-"])
print(mol)  # System(n_atoms=21, charge=0, ...)

# Minimize it (native MD engine, no subprocess) -> System with relaxed coords
mol = mol.minimize(steps=5000)

In [ ]:
# ============================================================
# The VSOMM case: 200 DISTINCT molecules, each minimized, then packed
# ============================================================
ga_sequences = [...]   # 200 different building-block lists from the GA

# Each is its own System (distinct topology + its own minimized coords)
mols = [ff.molecule(seq).minimize() for seq in ga_sequences]

# Lazy assembly of DISTINCT systems  (this is why '+' over distinct matters,
# not just '* N' over identical copies)
assembly = sum(mols)                       # Assembly(200 distinct molecules)

# Pack + solvate + neutralize in one chain
system = (assembly
          .pack(density=900, seed=42)      # ran_box places the 200 molecules
          .solvate(ff.solvent("SPC"), density=900)
          .neutralize("Na+"))

print(system)  # the whole solvated, neutralized box as ONE System

🔴 **DECISION D5 — does `sum()` of Systems work?**
For `sum(mols)` to work, `System + System` must return an `Assembly` (lazy, Model B). This is the natural fit for the VSOMM workload (distinct molecules, each with coords). Confirm this is a first-class path, not an afterthought.

🔴 **DECISION D6 — split or not?**
Notice `ff.molecule(seq)` returns a `System` (topo+coords) because `make_top`+coordinate-pipeline run together. But you also said people might want **topology only** or **coordinates only**. Options:
- (a) `ff.molecule(seq)` returns a `System` (topo+coords), always.
- (b) `ff.topology(seq)` returns a `Topology` only (just make_top); `.build_coordinates()` adds coords later; `System(topo, conf)` combines.

*My lean: offer both.* `ff.topology(seq)` for the topology-only crowd; `ff.molecule(seq)` = `ff.topology(seq).build_coordinates()` as the convenience that returns a full `System`.

---
## Part 6 — Running MD (`InputParameters`, reporters)

The native, clean half. No `.imd` files needed (your earlier Q1 answer).

In [ ]:
# Native parameter object — factory methods, no file
params = gromos.InputParameters.nvt(dt=0.002, steps=100_000, temperature=300)
params.cutoff = 1.4          # setters for individual knobs

# Old-fashioned path still works (your request):
params = gromos.InputParameters.from_file("run.imd")

# A Simulation binds a System + parameters
sim = gromos.Simulation(system, params)

# Reporters stream data out (OpenMM-style) — no .tre file needed
energy = gromos.EnergyReporter(every=100)
sim.reporters.append(energy)

sim.run(100_000)            # runs; reporter collects as it goes

df = energy.to_timeseries().to_dataframe()
df["total"].plot()

🔴 **DECISION D7 (your Q8) — can you mutate `params` after `Simulation(system, params)`?**
- (a) **freeze**: mutating after binding raises a clear error (safe, simple)
- (b) **live**: `params.temperature = 350` changes the running sim (OpenMM-like, more work)

*My lean: (a) freeze now.* The footgun (silent no-op) is worse than a clear error.

🔴 **DECISION D8 — does `sim.run()` mutate `system` or return a new one?**
- (a) mutate in place: `system.positions` updated after `run()`
- (b) `result = sim.run(...)`; `result.system` is the new state, original untouched

*My lean: (b)* — immutable-ish, chainable, matches `MDResult`.

---
## Part 7 — Full picture, one screen

If all the leans above hold, the whole thing reads like this:

In [ ]:
import gromos

ff = gromos.ForceField("54a7")

# build (subprocess binaries in background)
mols     = [ff.molecule(seq).minimize() for seq in ga_sequences]
assembly = sum(mols)
system   = (assembly
            .pack(density=900, seed=42)
            .solvate(ff.solvent("SPC"), density=900)
            .neutralize("Na+"))

# run (native engine, no subprocess, no .imd files)
params = gromos.InputParameters.nvt(dt=0.002, steps=1_000_000, temperature=300)
sim    = gromos.Simulation(system, params)
sim.reporters.append(gromos.EnergyReporter(every=1000))
result = sim.run(1_000_000)

result.energies.to_dataframe().write_csv("energies.csv")
result.system.write("final.cnf")

---
## Open decisions — answer these and I'll lock the plan

| # | Question | My lean |
|---|---|---|
| **D1** | `select()` returns indices or sub-System? | indices |
| **D2** | `+` eager or lazy? | **lazy** (Assembly plan, pack at end) |
| **D3** | what is `* N`? | count in the plan (needs lazy) |
| **D4** | `.solvate(n=)` vs `.solvate(density=)` | exactly one, both = error |
| **D5** | `sum(distinct systems)` first-class? | yes |
| **D6** | `ff.topology(seq)` AND `ff.molecule(seq)`? | both |
| **D7** | mutate `params` after binding? | freeze |
| **D8** | `run()` mutate vs return new? | return new |

**The load-bearing one is D2 (lazy vs eager `+`).** Everything else follows from it. If you say “lazy,” the `Assembly` concept appears and the whole algebra becomes coherent. If “eager,” we drop `+`/`*` and make solvate/pack the only assembly verbs.

Once you answer, I'll fold this into PLAN.md §P3 as the `System`-centric object model, the lazy-assembly algebra, and the two maturity tiers (native run / subprocess build).